<a href="https://colab.research.google.com/github/mateusbasilio53-cmd/clinia/blob/main/clinia_dados_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Base_ficticia_DM2_CLINIA.xlsx to Base_ficticia_DM2_CLINIA.xlsx


In [ ]:
import pandas as pd

# ============================================================
# 1. CARREGAR OS DADOS
# ============================================================
df = pd.read_excel('Base_ficticia_DM2_CLINIA.xlsx')

print('Shape original:', df.shape)
print()

# ============================================================
# 2. DIAGNÓSTICO — padronizar variações
# ============================================================
df['diagnostico'] = df['diagnostico'].replace({
    'DM tipo II': 'diabetes_tipo2',
    'Diabetes tipo 2': 'diabetes_tipo2',
    'Diabetes mellitus tipo II': 'diabetes_tipo2',
    'Diabetes mellitus tipo 2': 'diabetes_tipo2',
    'DM2': 'diabetes_tipo2'
})

print('=== DIAGNÓSTICO ===')
print(df['diagnostico'].value_counts())
print()

# ============================================================
# 3. MEDICAMENTOS — padronizar texto para minúsculo
# ============================================================
df['medicamentos'] = df['medicamentos'].str.lower().str.strip()

print('=== MEDICAMENTOS (primeiras linhas) ===')
print(df['medicamentos'].head(5))
print()

# ============================================================
# 4. NEOPLASIA E GESTAÇÃO — padronizar Sim/Não
# ============================================================
df['neoplasia_ativa'] = df['neoplasia_ativa'].str.strip()
df['gestacao'] = df['gestacao'].str.strip()

print('=== NEOPLASIA ===')
print(df['neoplasia_ativa'].value_counts())
print()
print('=== GESTAÇÃO ===')
print(df['gestacao'].value_counts())
print()

# ============================================================
# 5. ELEGIBILIDADE — padronizar texto
# ============================================================
df['elegibilidade_calculada'] = df['elegibilidade_calculada'].str.strip()

print('=== ELEGIBILIDADE ===')
print(df['elegibilidade_calculada'].value_counts())
print()

# ============================================================
# 6. VERIFICAÇÃO FINAL
# ============================================================
print('=== SHAPE FINAL ===')
print(df.shape)
print()
print('=== NULOS ===')
print(df.isnull().sum())
print()
print('=== TIPOS DE DADOS ===')
print(df.dtypes)

# ============================================================
# 7. SALVAR BASE TRATADA
# ============================================================
df.to_csv('CLINIA_base_tratada.csv', index=False)
print()
print('Base salva com sucesso!')

Shape original: (200, 17)

=== DIAGNÓSTICO ===
diagnostico
diabetes_tipo2    200
Name: count, dtype: int64

=== MEDICAMENTOS (primeiras linhas) ===
0                           metformina 1.000 mg 2x/dia
1    metformina xr 500 mg 2x/dia; levotiroxina 75 m...
2    metformina 1.000 mg 2x/dia; losartana 50 mg/di...
3    metformina 850 mg 2x/dia; sinvastatina 20 mg à...
4    metformina xr 500 mg 2x/dia; sinvastatina 20 m...
Name: medicamentos, dtype: object

=== NEOPLASIA ===
neoplasia_ativa
Não    192
Sim      8
Name: count, dtype: int64

=== GESTAÇÃO ===
gestacao
Não    192
Sim      8
Name: count, dtype: int64

=== ELEGIBILIDADE ===
elegibilidade_calculada
Elegível        115
Não elegível     85
Name: count, dtype: int64

=== SHAPE FINAL ===
(200, 17)

=== NULOS ===
id_paciente                0
idade                      0
sexo                       0
diagnostico                0
diagnostico_secundario     0
hba1c                      0
creatinina                 0
glicemia_jejum         

In [ ]:
import re  # biblioteca para manipular padrões dentro de texto

def limpar_medicamentos(texto):

    # coloca tudo em minúsculo para padronizar
    texto = texto.lower()

    # remove qualquer sequência de números — ex: 850, 50, 20, 75
    texto = re.sub(r'\d+', '', texto)

    # remove palavras que não são medicamentos
    # \b garante que remove a palavra inteira, não parte de outra
    palavras_remover = [
        'mg',    # unidade de medida
        'mcg',   # unidade de medida
        'xr',    # tipo de formulação (liberação prolongada)
        'dia',   # frequência
        'noite', # frequência
        'manhã', # frequência
        'x',     # frequência (como em 2x)
        'à',     # preposição
        'ao',    # preposição
        'mr',    # tipo de formulação
    ]
    for palavra in palavras_remover:
        texto = re.sub(r'\b' + palavra + r'\b', '', texto)

    # remove espaços extras que sobraram após as remoções
    texto = re.sub(r'\s+', ' ', texto).strip()

    return texto

# aplica a função em cada linha da coluna medicamentos
df['medicamentos'] = df['medicamentos'].apply(limpar_medicamentos)

# verifica o resultado
print(df['medicamentos'].head(10))

0                                  metformina . /
1                    metformina /; levotiroxina /
2    metformina . /; losartana /; atorvastatina /
3                      metformina /; sinvastatina
4                      metformina /; sinvastatina
5     metformina e ; losartana /; atorvastatina /
6                   metformina . ; levotiroxina /
7                      metformina /; amlodipino /
8                   metformina . ; levotiroxina /
9                                  metformina . /
Name: medicamentos, dtype: object


In [ ]:
df


In [ ]:
# pegar 5 elegíveis, 5 não elegíveis e 3 limítrofes
amostra_elegivel     = df[df['elegibilidade_calculada'] == 'Elegível'].head(5)
amostra_nao_elegivel = df[df['elegibilidade_calculada'] == 'Não elegível'].head(5)
amostra_limitrofe    = df[df['grupo_simulacao'] == 'Limítrofe'].head(3)

# juntar os três grupos
amostra = pd.concat([amostra_elegivel, amostra_nao_elegivel, amostra_limitrofe])

# verificar
print(amostra.shape)
amostra

(13, 17)


,id_paciente,idade,sexo,diagnostico,diagnostico_secundario,hba1c,creatinina,glicemia_jejum,imc,medicamentos,neoplasia_ativa,gestacao,data_atendimento,anotacao_medica,grupo_simulacao,marcador_caso,elegibilidade_calculada
0,PAC-001,36,M,diabetes_tipo2,Sem comorbidades relevantes,9.5,0.87,171,32.6,metformina . /,Não,Não,2026-04-10,Em consulta de rotina para controle metabólico...,Elegível,Critérios claramente atendidos,Elegível
1,PAC-002,47,F,diabetes_tipo2,Hipertensão arterial sistêmica; Dislipidemia,8.3,0.65,178,35.3,metformina /; levotiroxina /,Não,Não,2026-04-12,Reavaliação clínica de paciente com diabetes t...,Elegível,Critérios claramente atendidos,Elegível
2,PAC-003,35,F,diabetes_tipo2,Doença do refluxo gastroesofágico,9.3,0.96,194,36.5,metformina . /; losartana /; atorvastatina /,Não,Não,2026-04-21,Retorna para seguimento de diabetes mellitus t...,Elegível,Critérios claramente atendidos,Elegível
3,PAC-004,43,F,diabetes_tipo2,Hipertensão arterial sistêmica; Dislipidemia,9.9,1.20,172,40.5,metformina /; sinvastatina,Não,Não,2026-07-14,"Paciente com diagnóstico prévio de DM2, em seg...",Elegível,Critérios claramente atendidos,Elegível
4,PAC-005,43,M,diabetes_tipo2,Obesidade; Hipertensão arterial sistêmica,9.7,1.10,177,36.7,metformina /; sinvastatina,Não,Não,2026-03-21,Em consulta de rotina para controle metabólico...,Elegível,Critérios claramente atendidos,Elegível
100,PAC-101,82,F,diabetes_tipo2,Obesidade,9.2,1.08,178,29.3,metformina . ; atorvastatina,Não,Não,2026-01-19,Retorna para seguimento de diabetes mellitus t...,Não elegível,Idade acima da faixa,Não elegível
101,PAC-102,36,M,diabetes_tipo2,Esteatose hepática,10.8,1.27,186,31.9,metformina .,Não,Não,2026-01-19,Reavaliação clínica de paciente com diabetes t...,Não elegível,HbA1c acima da faixa,Não elegível
102,PAC-103,58,M,diabetes_tipo2,Esteatose hepática,10.0,1.85,178,27.0,metformina . ; amlodipino /,Não,Não,2026-03-01,Reavaliação clínica de paciente com diabetes t...,Não elegível,Creatinina acima do limite,Não elegível
103,PAC-104,54,F,diabetes_tipo2,Dislipidemia,8.2,1.19,182,38.4,dapagliflozina /; gliclazida /,Não,Não,2026-05-04,Reavaliação clínica de paciente com diabetes t...,Não elegível,Sem uso de Metformina,Não elegível
104,PAC-105,51,M,diabetes_tipo2,Dislipidemia,8.5,1.31,188,28.9,metformina /; atorvastatina,Sim,Não,2026-02-01,Paciente em acompanhamento ambulatorial por DM...,Não elegível,Neoplasia ativa,Não elegível


In [ ]:
import os

drive_path = '/content/drive/MyDrive/Colab Notebooks/clinia/'

# Create the directory if it doesn't exist
os.makedirs(drive_path, exist_ok=True)

# salvar amostra para reunião no Drive
amostra.to_excel(drive_path + 'CLINIA_amostra_reuniao.xlsx', index=False)

# salvar base completa para treinar depois no Drive
df.to_excel(drive_path + 'CLINIA_base_completa_tratada.xlsx', index=False)

print(f"Arquivos salvos no Google Drive em: {drive_path}")

# baixar os arquivos também para o seu computador (opcional)
from google.colab import files
files.download(drive_path + 'CLINIA_amostra_reuniao.xlsx')
files.download(drive_path + 'CLINIA_base_completa_tratada.xlsx')

Arquivos salvos no Google Drive em: /content/drive/MyDrive/Colab Notebooks/clinia/


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# montar o drive se ainda não montou
from google.colab import drive
drive.mount('/content/drive')

# salvar base completa no Drive
/CLINIA/CLINIA_base_compledf.to_excel('/content/drive/MyDriveta_tratada.xlsx', index=False)

# salvar amostra no Drive
amostra.to_excel('/content/drive/MyDrive/CLINIA/CLINIA_amostra_reuniao.xlsx', index=False)

print('Salvo no Drive com sucesso!')

ValueError: Mountpoint must not already contain files

In [ ]:
df

,id_paciente,idade,sexo,diagnostico,diagnostico_secundario,hba1c,creatinina,glicemia_jejum,imc,medicamentos,neoplasia_ativa,gestacao,data_atendimento,anotacao_medica,grupo_simulacao,marcador_caso,elegibilidade_calculada
0,PAC-001,36,M,diabetes_tipo2,Sem comorbidades relevantes,9.5,0.87,171,32.6,metformina . /,Não,Não,2026-04-10,Em consulta de rotina para controle metabólico...,Elegível,Critérios claramente atendidos,Elegível
1,PAC-002,47,F,diabetes_tipo2,Hipertensão arterial sistêmica; Dislipidemia,8.3,0.65,178,35.3,metformina /; levotiroxina /,Não,Não,2026-04-12,Reavaliação clínica de paciente com diabetes t...,Elegível,Critérios claramente atendidos,Elegível
2,PAC-003,35,F,diabetes_tipo2,Doença do refluxo gastroesofágico,9.3,0.96,194,36.5,metformina . /; losartana /; atorvastatina /,Não,Não,2026-04-21,Retorna para seguimento de diabetes mellitus t...,Elegível,Critérios claramente atendidos,Elegível
3,PAC-004,43,F,diabetes_tipo2,Hipertensão arterial sistêmica; Dislipidemia,9.9,1.20,172,40.5,metformina /; sinvastatina,Não,Não,2026-07-14,"Paciente com diagnóstico prévio de DM2, em seg...",Elegível,Critérios claramente atendidos,Elegível
4,PAC-005,43,M,diabetes_tipo2,Obesidade; Hipertensão arterial sistêmica,9.7,1.10,177,36.7,metformina /; sinvastatina,Não,Não,2026-03-21,Em consulta de rotina para controle metabólico...,Elegível,Critérios claramente atendidos,Elegível
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,PAC-196,60,F,diabetes_tipo2,Doença do refluxo gastroesofágico,10.5,1.25,191,37.9,metformina /; losartana /; atorvastatina /,Não,Não,2026-06-05,Retorna para seguimento de diabetes mellitus t...,Limítrofe,HbA1c máxima — dentro por pequena margem,Elegível
196,PAC-197,55,F,diabetes_tipo2,Doença do refluxo gastroesofágico,7.1,1.18,152,27.7,metformina e ; losartana /; atorvastatina /,Não,Não,2026-06-09,Em consulta de rotina para controle metabólico...,Limítrofe,HbA1c mínima — dentro por pequena margem,Elegível
197,PAC-198,39,M,diabetes_tipo2,Hipertensão arterial sistêmica,8.1,0.90,166,25.0,metformina . /; atorvastatina,Não,Não,2026-04-06,Em consulta de rotina para controle metabólico...,Limítrofe,IMC mínimo — dentro por pequena margem,Elegível
198,PAC-199,69,M,diabetes_tipo2,Obesidade,8.0,1.23,174,37.8,metformina e ; losartana /; atorvastatina /,Não,Não,2026-04-09,Paciente em acompanhamento ambulatorial por DM...,Limítrofe,Idade máxima — dentro por pequena margem,Elegível


In [ ]:
df.to_excel("Base_ficticia_DM2_CLINIA.xlsx", index=False)


In [ ]:
from google.colab import files
files.download(drive_path + 'CLINIA_amostra_reuniao.xlsx')
files.download(drive_path + 'CLINIA_base_completa_tratada.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download(drive_path + 'CLINIA_base_completa_tratada.xlsx')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df.to_excel(drive_path + 'CLINIA_base_completa_tratada.xlsx', index=False)

In [ ]:
files.download(drive_path + 'CLINIA_base_completa_tratada.xlsx')